In [ ]:
# As this code need to be conneceted with Thingspeak, so we need to login to thingspeak
# if you want to use my user name and password for thingspeak it is available here
# user name: Ikhlaq-R@ulster.ac.uk
# Password: Ra39220077
# Simulation -> ThingSpeak Live Stream -> Local Logs -> Fetch -> Analysis -> Plots -> ZIP
# Field mapping:
#   Field1 = Voltage, Field2 = Current, Field3 = Power, Field4 = Flag (0=normal,1=spike)
# Notes:
# - ThingSpeak often enforces ~15s minimum update interval (free tier), so we use 15s.
# - Faster "visual change" is achieved by increasing noise and speeding up the cycle.

!pip -q install requests pandas matplotlib

import os              # It is used for folder/file operations
import time            # UIt is used for delay between ThingSpeak uploads
import math            # It is used for sine wave simulation
import random          # It is used for random noise, spikes, and dropouts
import zipfile         # It is used to create ZIP file of results
from datetime import datetime, timezone  # It is used for timestamps
import requests        # It is used to send/receive data from ThingSpeak API
import pandas as pd    # It is used for data handling and CSV files
import matplotlib.pyplot as plt  # It is used for plotting graphs
# USER SETTINGS
CHANNEL_ID    = "3254566"  # It is ThingSpeak Channel ID
WRITE_API_KEY = "SW5N7H22MBPH2EN5" # Write API key is used to send/upload data to ThingSpeak
READ_API_KEY  = "HZRV6OOT5VH1UWRF" # Read API key is used to fetch/read data from ThingSpeak

INTERVAL_SEC = 15            # ThingSpeak free tier usually requires about 15 seconds gap between uploads
POINTS_PER_SCENARIO = 200 # Number of readings generated for each scenario
SCENARIOS = ["normal", "high", "low", "fault"]

# ThingSpeak endpoints
THINGSPEAK_UPDATE_URL = "https://api.thingspeak.com/update.json" # This URL used for uploading data to ThingSpeak
THINGSPEAK_READ_URL   = "https://api.thingspeak.com/channels/{channel_id}/feeds.json" # This URL used for fetching stored data from ThingSpeak

# Helpers

def ensure_dir(path: str):
    if path and not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def hour_local():
    return datetime.now().hour

def ts_utc_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

# -----------------------
# Upload (handles ThingSpeak response types)
# -----------------------
def upload_to_thingspeak(write_key, voltage, current, power, flag, timeout=12):
    payload = {
        "api_key": write_key,
        "field1": f"{voltage:.3f}",
        "field2": f"{current:.3f}",
        "field3": f"{power:.3f}",
        "field4": str(int(flag)),
    }
    try:
        r = requests.post(THINGSPEAK_UPDATE_URL, data=payload, timeout=timeout)

        if r.status_code != 200:
            return False, f"HTTP {r.status_code}: {r.text[:120]}"

        # ThingSpeak may return dict/int/plain text
        try:
            j = r.json()
            if isinstance(j, dict):
                entry_id = j.get("entry_id")
                if not entry_id or str(entry_id) == "0":
                    return False, "entry_id=0 (rejected)"
                return True, f"entry_id={entry_id}"
            if isinstance(j, int):
                return (False, "entry_id=0 (rejected)") if j == 0 else (True, f"entry_id={j}")
        except Exception:
            pass

        txt = r.text.strip()
        if txt.isdigit():
            return (False, "entry_id=0 (rejected)") if txt == "0" else (True, f"entry_id={txt}")

        return False, f"HTTP 200 but unexpected response: {txt[:120]}"

    except requests.RequestException as e:
        return False, f"Request error: {e}"

# Simulation Function
def generate_reading(scenario, state):
    # Simulate voltage around 230V with small random variation
    voltage = 230.0 + random.uniform(-2.5, 2.5)

    # Base current depending on scenario
    base_current = 1.0
    if scenario == "high":
        base_current *= 1.6
    elif scenario == "low":
        base_current *= 0.7
    elif scenario == "fault":
        base_current *= 1.1

    # Time-of-day bump (evening use)
    hr = hour_local()
    if 18 <= hr < 22:
        base_current += 0.9

    # Trend component (faster so it looks more "live")
    t = time.time()
    base_current += 0.35 * math.sin(t / 90.0)

    # Simulate appliance ON/OFF cycling
    cycle = math.sin(t / 45.0)
    cycle_on = 1.0 if cycle > 0.4 else 0.0
    cycle_current = 0.7 * cycle_on

    # Noise (increased so graphs move more)
    current = base_current + cycle_current + random.uniform(-0.35, 0.35)

    # Spike events (kettle/microwave style)
    flag = 0
    if random.random() < 0.06:
        flag = 1
        current += random.uniform(3.0, 8.0)

    # Fault: occasional prolonged high draw
    if scenario == "fault":
        fault_timer = state.get("fault_timer", 0)
        if fault_timer > 0:
            flag = 1
            current += 4.0
            state["fault_timer"] = fault_timer - 1
        else:
            if random.random() < 0.02:
                state["fault_timer"] = random.randint(6, 16)

    voltage = clamp(voltage, 215.0, 245.0)
    current = clamp(current, 0.05, 20.0)
    power = voltage * current

    return voltage, current, power, flag
# -----------------------
# Analysis outputs
# -----------------------
def make_plots_and_outputs(df, out_dir, scenario):
    ensure_dir(out_dir)
    df = df.copy()
    df["ts"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["ts"]).sort_values("ts")
    df["power_roll_mean"] = df["power"].rolling(window=10, min_periods=1).mean()
    # Power time-series
    plt.figure()
    plt.plot(df["ts"], df["power"], label="Power (W)")
    plt.plot(df["ts"], df["power_roll_mean"], label="Rolling mean")
    plt.title(f"Power Over Time ({scenario})")
    plt.xlabel("Time (UTC)")
    plt.ylabel("Power (W)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{scenario}_power_timeseries.png"))
    plt.close()

    # Hourly profile
    df["hour"] = df["ts"].dt.hour
    hourly = df.groupby("hour")["power"].mean().reset_index()

    plt.figure()
    plt.plot(hourly["hour"], hourly["power"], marker="o")
    plt.title(f"Average Power by Hour ({scenario})")
    plt.xlabel("Hour of day")
    plt.ylabel("Average Power (W)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{scenario}_hourly_profile.png"))
    plt.close()

    # Peaks
    peaks = df.nlargest(10, "power")[["ts", "power", "voltage", "current", "flag"]]
    peaks.to_csv(os.path.join(out_dir, f"{scenario}_top_peaks.csv"), index=False)

    # Rule-based unusual points
    mean_p = df["power"].mean()
    std_p = df["power"].std() if pd.notna(df["power"].std()) else 0.0
    threshold = mean_p + 2 * std_p

    df["unusual_rule"] = (df["power"] > threshold).astype(int)
    df[df["unusual_rule"] == 1][["ts", "power", "voltage", "current", "flag"]].to_csv(
        os.path.join(out_dir, f"{scenario}_unusual_points.csv"), index=False
    )

    # Summary text
    with open(os.path.join(out_dir, f"{scenario}_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Scenario: {scenario}\n")
        f.write(f"Rows: {len(df)}\n")
        f.write(f"Start (UTC): {df['ts'].min()}\n")
        f.write(f"End (UTC): {df['ts'].max()}\n")
        f.write(f"Mean Power (W): {mean_p:.2f}\n")
        f.write(f"Max Power (W): {df['power'].max():.2f}\n")
        f.write(f"Spike Count: {(df['flag']==1).sum()}\n")
        f.write(f"Rule Threshold (mean+2std): {threshold:.2f}\n")

# Run one scenario

def run_scenario(scenario):
    ensure_dir("results/scenarios")
    out_dir = f"results/scenarios/{scenario}"
    ensure_dir(out_dir)

    state = {} # State is used to remember fault duration
    rows = [] # rows stores local readings
    dropouts = 0 # Count simulated missing readings
    sent_ok = 0   # Count successful ThingSpeak uploads

    print(f"\n=== Scenario: {scenario} ===")
    for i in range(POINTS_PER_SCENARIO):
        # Simulated dropout (skips upload)
        if random.random() < 0.02: # Simulate communication dropout with 2% probability
            dropouts += 1
            print(f"[{datetime.now().isoformat(timespec='seconds')}] Dropout simulated")
            time.sleep(INTERVAL_SEC)
            continue

        v, c, p, flag = generate_reading(scenario, state)
        ts = ts_utc_iso()

        rows.append({
            "timestamp_utc": ts,
            "voltage": round(v, 3),
            "current": round(c, 3),
            "power": round(p, 3),
            "flag": int(flag),
            "scenario": scenario
        })

        ok, msg = upload_to_thingspeak(WRITE_API_KEY, v, c, p, flag)
        if ok:
            sent_ok += 1

        print(f"[{datetime.now().isoformat(timespec='seconds')}] {'SENT' if ok else 'FAIL'} "
              f"V={v:6.2f} I={c:5.2f} P={p:7.1f} Flag={flag} ({msg})")

        time.sleep(INTERVAL_SEC)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, f"{scenario}.csv"), index=False)
    make_plots_and_outputs(df, out_dir, scenario)

    attempts = len(rows)
    upload_success = (sent_ok / attempts * 100.0) if attempts else 0.0
    missing_rate = (dropouts / POINTS_PER_SCENARIO * 100.0) if POINTS_PER_SCENARIO else 0.0

    return {
        "scenario": scenario,
        "rows_logged": attempts,
        "dropouts": dropouts,
        "missing_rate_percent": round(missing_rate, 2),
        "upload_success_percent": round(upload_success, 2),
        "mean_power": round(df["power"].mean(), 2) if attempts else 0,
        "max_power": round(df["power"].max(), 2) if attempts else 0,
        "spike_count": int((df["flag"] == 1).sum()) if attempts else 0,
    }

# -----------------------
# Fetch from ThingSpeak
# -----------------------
def fetch_from_thingspeak():
    ensure_dir("results/cloud_fetch")
    url = THINGSPEAK_READ_URL.format(channel_id=CHANNEL_ID)
    params = {"results": 8000}
    if READ_API_KEY and READ_API_KEY.strip():
        params["api_key"] = READ_API_KEY.strip()

    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    feeds = data.get("feeds", [])
    out_csv = "results/cloud_fetch/fetched_from_thingspeak.csv"
    out_rows = []
    for item in feeds:
        out_rows.append({
            "created_at": item.get("created_at", ""),
            "voltage": item.get("field1", ""),
            "current": item.get("field2", ""),
            "power": item.get("field3", ""),
            "flag": item.get("field4", ""),
        })
    pd.DataFrame(out_rows).to_csv(out_csv, index=False)
    print(f"\nFetched {len(out_rows)} rows to {out_csv}")

# -----------------------
# Main run
# -----------------------
ensure_dir("results")
ensure_dir("results/scenarios")

print("Starting single-cell run...")
print("Channel:", CHANNEL_ID)
print("Interval (sec):", INTERVAL_SEC)
print("Points per scenario:", POINTS_PER_SCENARIO)
print("Scenarios:", SCENARIOS)

summaries = []
for sc in SCENARIOS:
    summaries.append(run_scenario(sc))

summary_df = pd.DataFrame(summaries)
summary_df.to_csv("results/scenario_summary_table.csv", index=False)
print("\nSaved results/scenario_summary_table.csv")
print(summary_df)

# Combined comparison plot for paper
plt.figure()
for sc in SCENARIOS:
    df = pd.read_csv(f"results/scenarios/{sc}/{sc}.csv")
    df["ts"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["ts"]).sort_values("ts")
    df_plot = df.iloc[::max(1, len(df)//200)]
    plt.plot(df_plot["ts"], df_plot["power"], label=sc)

plt.title("Power Comparison Across Scenarios (sampled)")
plt.xlabel("Time (UTC)")
plt.ylabel("Power (W)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("results/power_comparison_across_scenarios.png")
plt.close()
print("Saved results/power_comparison_across_scenarios.png")

# Fetch from cloud for proof (optional)
try:
    fetch_from_thingspeak()
except Exception as e:
    print("\nCloud fetch skipped/failed:", e)

# Zip everything for download
zip_name = "project_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("results"):
        for fn in files:
            fp = os.path.join(root, fn)
            z.write(fp, arcname=fp)
print(f"\nCreated {zip_name} (download from Colab files)")

# Quick display of the comparison plot
try:
    from PIL import Image
    img = Image.open("results/power_comparison_across_scenarios.png")
    plt.figure(figsize=(12,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
except Exception as e:
    print("Preview failed:", e)




Starting single-cell run...
Channel: 3254566
Interval (sec): 15
Points per scenario: 200
Scenarios: ['normal', 'high', 'low', 'fault']

=== Scenario: normal ===
[2026-04-26T17:29:15] FAIL V=228.13 I= 1.41 P=  320.9 Flag=0 (entry_id=0 (rejected))
[2026-04-26T17:29:30] FAIL V=228.62 I= 1.70 P=  387.5 Flag=0 (entry_id=0 (rejected))
[2026-04-26T17:29:45] Dropout simulated
[2026-04-26T17:30:00] FAIL V=229.67 I= 1.75 P=  402.9 Flag=0 (entry_id=0 (rejected))
[2026-04-26T17:30:15] FAIL V=229.29 I= 2.16 P=  495.2 Flag=0 (entry_id=0 (rejected))
